# Predictive Maintenance: AI4I 2020
**Goal:** Predict machine failure with high precision and useful recall to reduce unplanned downtime.

In [ ]:
# --- Core ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Modeling / split ---
from sklearn.model_selection import train_test_split

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

# --- Metrics / plots ---
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    PrecisionRecallDisplay,
)

# Reproducibility
RANDOM_STATE = 42

# Project paths
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True, parents=True)
ASSETS_DIR = Path("assets"); ASSETS_DIR.mkdir(exist_ok=True, parents=True)

## Data Overview
- Load dataset and verify shape/columns
- Check target balance and missing values

In [ ]:
# Load data (place ai4i2020.csv in the data/ folder)
csv_path = DATA_DIR / 'ai4i2020.csv'
assert csv_path.exists(), f"Missing {csv_path}. Download from UCI and place it in data/."
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

In [ ]:
# --- Basic Data Checks ---

# View column names and dataset dimensions
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumns:\n", df.columns.tolist())

# Summary statistics for numeric features
display(df.describe().T.style.background_gradient(cmap="Blues"))

In [ ]:
# Define target column (based on dataset description)
target_col = "Machine failure"

# Check class distribution
print("Target distribution (proportion):")
print(df[target_col].value_counts(normalize=True).rename("proportion").round(3))

In [ ]:
# Check for missing values
print("Missing values by column:")
print(df.isna().sum())

In [ ]:
# Label consistency: 'Machine failure' vs OR(TWF,HDF,PWF,OSF,RNF)
flags = ["TWF", "HDF", "PWF", "OSF", "RNF"]
has_subtype = (df[flags].sum(axis=1) > 0).astype(int)

inconsistent_mask = df["Machine failure"].astype(int) != has_subtype
n_inconsistent = int(inconsistent_mask.sum())
n_total = len(df)
pct_inconsistent = 100 * n_inconsistent / n_total

print(f"{n_inconsistent:,} out of {n_total:,} rows "
      f"({pct_inconsistent:.2f}%) where 'Machine failure' != OR(TWF,HDF,PWF,OSF,RNF)")

> **Data quality check:**  
> Verified that the overall `Machine failure` flag aligns with the sub-failure types (`TWF–RNF`).  
> Only 27 out of 10,000 rows showed minor inconsistencies, treated as negligible label noise.


## Feature Engineering
`Product ID` begins with a letter (L/M/H) that encodes product category.
- Extract `Product_Category` from `Product ID`
- Remove leakage columns: `TWF, HDF, PWF, OSF, RNF`

In [ ]:
# Extract product category (L/M/H) from 'Product ID' and drop the raw ID
if "Product ID" in df.columns:
    df["Product_Category"] = df["Product ID"].astype(str).str[0]
    df.drop(columns=["Product ID"], inplace=True)

# Quick check
if "Product_Category" in df.columns:
    print("Product_Category counts:")
    print(df["Product_Category"].value_counts(dropna=False))

In [ ]:
# --- Features (X) and target (y) ---

# Drop pure row identifiers (keep only those that actually exist)
id_cols = ["UDI"]
id_cols = [c for c in id_cols if c in df.columns]

# Feature columns = everything except target + ids
feature_cols = [c for c in df.columns if c not in [target_col] + id_cols]

# Build X and y (keep X as DataFrame to preserve column names)
X = df[feature_cols]
y = df[target_col].astype(int)

print(f"Target: {target_col}")
print(f"Number of features: {len(feature_cols)} columns")
print("First few features:", feature_cols[:8])


In [ ]:
# Remove failure subtype columns from features to avoid leakage
leak_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
leak_cols_present = [c for c in leak_cols if c in feature_cols]
print("Dropping potential leakage columns:", leak_cols_present)

feature_cols = [c for c in feature_cols if c not in leak_cols_present]
X = df[feature_cols]  # refresh X after updating feature_cols

print(f"Final feature count: {len(feature_cols)}")
print("Sample features:", feature_cols[:8])

## Split & Preprocessing
- Stratified train/validation/test split (60% / 15% / 25%)
- Use validation data to select the operating threshold; reserve test data for final evaluation
- Numerical scaling and categorical one-hot encoding

In [ ]:
# Column types
num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

# First reserve the final test set (25%); do not use it to choose a threshold.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

# Split the remaining 75% into 60% train and 15% validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval,
    random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

In [ ]:
# One shared version-safe OneHotEncoder
try:
    ohe_shared = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_shared = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Logistic Regression: scale numerics + OHE categoricals
preprocess_for_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", ohe_shared, cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Tree models: passthrough numerics + OHE categoricals (no scaling)
preprocess_for_trees = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", ohe_shared, cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Sanity check: verify transformed feature matrix shape
preprocess_for_lr.fit_transform(X_train).shape

## Models

### Calibrated Logistic Regression
- `class_weight='balanced'` to handle imbalance
- Platt scaling with cross-validated calibration

In [ ]:
# Calibrated Logistic Regression (class-balanced)
logit_pipe = Pipeline([
    ("pre", preprocess_for_lr),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

# Version-safe CalibratedClassifierCV
try:
    cal_logit = CalibratedClassifierCV(estimator=logit_pipe, cv=3, method="sigmoid")
except TypeError:
    cal_logit = CalibratedClassifierCV(base_estimator=logit_pipe, cv=3, method="sigmoid")

cal_logit.fit(X_train, y_train)
proba_lr_val = cal_logit.predict_proba(X_val)[:, 1]
proba_lr = cal_logit.predict_proba(X_test)[:, 1]
print(
    f"Calibrated Logistic (test) | PR-AUC: {average_precision_score(y_test, proba_lr):.3f} | "
    f"ROC-AUC: {roc_auc_score(y_test, proba_lr):.3f}"
)

### Random Forest (Baseline)
- `class_weight='balanced'`
- Strong non-linear baseline

In [ ]:
# Random Forest baseline
rf_pipe = Pipeline([
    ("pre", preprocess_for_trees),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

rf_pipe.fit(X_train, y_train)
proba_rf_val = rf_pipe.predict_proba(X_val)[:, 1]
proba_rf = rf_pipe.predict_proba(X_test)[:, 1]
print(
    f"Random Forest (test) | PR-AUC: {average_precision_score(y_test, proba_rf):.3f} | "
    f"ROC-AUC: {roc_auc_score(y_test, proba_rf):.3f}"
)

## Evaluation & Curves

### Metrics Summary
- PR-AUC, ROC-AUC, and Recall at 90% Precision

In [ ]:
# Select thresholds on validation data, then report performance at those fixed thresholds on test data.
from sklearn.metrics import precision_score, recall_score

def threshold_for_precision(y_true, y_proba, precision_target=0.90):
    p, r, thresholds = precision_recall_curve(y_true, y_proba)
    valid = np.where(p[:-1] >= precision_target)[0]  # p/r align with thresholds[:-1]
    if len(valid) == 0:
        return np.nan
    best_idx = valid[np.argmax(r[:-1][valid])]
    return float(thresholds[best_idx])

P_TARGET = 0.90
thr_lr = threshold_for_precision(y_val, proba_lr_val, P_TARGET)
thr_rf = threshold_for_precision(y_val, proba_rf_val, P_TARGET)

def report_at_threshold(name, y_true, y_proba, threshold):
    if pd.isna(threshold):
        print(f"{name}: validation data did not reach {P_TARGET:.0%} precision; threshold unavailable.")
        return {"Model": name, "Precision@threshold": np.nan, "Recall@threshold": np.nan, "Threshold": np.nan}
    pred = (y_proba >= threshold).astype(int)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    print(f"{name} — TEST precision: {precision:.3f} | TEST recall: {recall:.3f} | validation-selected threshold: {threshold:.4f}")
    return {"Model": name, "Precision@threshold": precision, "Recall@threshold": recall, "Threshold": threshold}

operating_point_results = pd.DataFrame([
    report_at_threshold("Calibrated Logistic", y_test, proba_lr, thr_lr),
    report_at_threshold("Random Forest", y_test, proba_rf, thr_rf),
])
operating_point_results

In [ ]:
summary = pd.DataFrame([
    {"Model": "Calibrated Logistic", "PR-AUC": average_precision_score(y_test, proba_lr),
     "ROC-AUC": roc_auc_score(y_test, proba_lr)},
    {"Model": "Random Forest", "PR-AUC": average_precision_score(y_test, proba_rf),
     "ROC-AUC": roc_auc_score(y_test, proba_rf)},
]).round(4)

summary

In [ ]:
summary.to_csv(ASSETS_DIR / "metrics_summary.csv", index=False)

### Operating point evaluation
Thresholds are selected using the validation split only. Precision and recall below are measured on the untouched test split.

In [ ]:
# Save test-set operating-point metrics using validation-selected thresholds.
ASSETS_DIR.mkdir(exist_ok=True, parents=True)
operating_point_results.to_csv(ASSETS_DIR / "operating_point_metrics.csv", index=False)
operating_point_results

### Confusion Matrices
These matrices use the validation-selected thresholds and the held-out test split. If a model has no threshold meeting the validation precision target, its matrix uses the default 0.50 cutoff and is labeled accordingly.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def plot_confusion_matrix(y_true, y_proba, threshold, model_name, filename):
    # Use the validation-selected threshold when available; otherwise show the conventional 0.50 cutoff.
    used_threshold = 0.50 if pd.isna(threshold) else float(threshold)
    y_pred = (y_proba >= used_threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Failure", "Failure"])
    disp.plot(values_format="d", cmap="Blues")
    label = "default 0.50 cutoff" if pd.isna(threshold) else "validation-selected threshold"
    plt.title(f"{model_name} — Test Confusion Matrix ({label})")
    plt.tight_layout()
    plt.savefig(ASSETS_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()
    return cm

cm_lr = plot_confusion_matrix(y_test, proba_lr, thr_lr, "Calibrated Logistic Regression", "confusion_matrix_logistic.png")
cm_rf = plot_confusion_matrix(y_test, proba_rf, thr_rf, "Random Forest", "confusion_matrix_randomforest.png")

### Precision–Recall Curves
- Combined comparison plus individual curves

In [ ]:
# Precision–Recall curves

# Combined comparison
ax = plt.gca()
PrecisionRecallDisplay.from_predictions(y_test, proba_lr, name="Calibrated Logistic", ax=ax)
PrecisionRecallDisplay.from_predictions(y_test, proba_rf, name="Random Forest", ax=ax)
plt.title("Precision–Recall: Comparison")
plt.savefig(ASSETS_DIR / "pr_curve_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Individual: Random Forest
disp = PrecisionRecallDisplay.from_predictions(y_test, proba_rf, name="Random Forest")
plt.title("Precision–Recall: Random Forest")
plt.savefig(ASSETS_DIR / "pr_curve_randomforest.png", dpi=160, bbox_inches="tight")
plt.show()

# Individual: Calibrated Logistic
disp = PrecisionRecallDisplay.from_predictions(y_test, proba_lr, name="Calibrated Logistic")
plt.title("Precision–Recall: Calibrated Logistic")
plt.savefig(ASSETS_DIR / "pr_curve_logistic.png", dpi=160, bbox_inches="tight")
plt.show()


## Feature Importance

In [ ]:
# Random Forest top features
importances = rf_pipe.named_steps["clf"].feature_importances_
try:
    feature_names = rf_pipe.named_steps["pre"].get_feature_names_out()
except AttributeError:
    ohe = rf_pipe.named_steps["pre"].named_transformers_["cat"]
    cat_names = ohe.get_feature_names_out(cat_cols) if len(cat_cols) else np.array([])
    feature_names = np.concatenate([np.array(num_cols), cat_names])

top20 = (pd.Series(importances, index=feature_names)
           .sort_values(ascending=False)
           .head(20)
           .reset_index())
top20.columns = ["Feature", "Importance"]
top20.round(4)

In [ ]:
#Random Forest top features plot

top20.to_csv(ASSETS_DIR / "rf_feature_importances_top20.csv", index=False)
plt.figure(figsize=(8, 6))
plt.barh(top20["Feature"][::-1], top20["Importance"][::-1])
plt.title("Random Forest: Top 20 Feature Importances")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "feature_importances_rf.png", dpi=160, bbox_inches="tight")
plt.show()

### Feature Importance Insights

The Random Forest identifies **Torque**, **Rotational speed**, and **Tool wear** as dominant predictors of failure, aligning with real-world engineering intuition.  
Environmental factors such as **air** and **process temperature** also contribute, though less strongly.  
Categorical features (machine type and product category) have smaller impacts, mainly capturing contextual variation between equipment types.

## Conclusion
Compare PR-AUC and ROC-AUC on the held-out test set. The operating threshold is selected using validation data, then evaluated once on the test data to reduce threshold-selection bias. Random Forest and calibrated Logistic Regression provide different performance/interpretability trade-offs; results should be confirmed with repeated stratified cross-validation and, ideally, time-separated operational data before real-world use.